In [4]:
!pip install -q pandas openpyxl python-pptx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 12.6 MB/s eta 0:00:00


In [5]:
from google.colab import files

uploaded = files.upload()

Saving Dataset for Data Analytics (2).xlsx to Dataset for Data Analytics (2).xlsx


In [6]:
import pandas as pd
import sqlite3
from pptx import Presentation
from pptx.util import Inches, Pt
from google.colab import files

# Read Excel
file = list(uploaded.keys())[0]
df = pd.read_excel(file)

# Create SQL database
con = sqlite3.connect(":memory:")
df.to_sql("orders", con, index=False, if_exists="replace")

# SQL queries
product = pd.read_sql("""
SELECT Product,
COUNT(OrderID) AS Orders,
SUM(Quantity) AS Quantity,
SUM(TotalPrice) AS Revenue,
AVG(TotalPrice) AS AvgPrice
FROM orders
GROUP BY Product
ORDER BY Revenue DESC
""", con)

status = pd.read_sql("""
SELECT OrderStatus,
COUNT(OrderID) AS Orders,
SUM(TotalPrice) AS Revenue,
AVG(TotalPrice) AS AvgPrice
FROM orders
GROUP BY OrderStatus
ORDER BY Orders DESC
""", con)

payment = pd.read_sql("""
SELECT PaymentMethod,
COUNT(OrderID) AS Orders,
SUM(TotalPrice) AS Revenue,
AVG(TotalPrice) AS AvgPrice
FROM orders
GROUP BY PaymentMethod
ORDER BY Revenue DESC
""", con)

referral = pd.read_sql("""
SELECT ReferralSource,
COUNT(OrderID) AS Orders,
SUM(TotalPrice) AS Revenue,
AVG(TotalPrice) AS AvgPrice
FROM orders
GROUP BY ReferralSource
ORDER BY Revenue DESC
""", con)

# Create PPT
ppt = Presentation()

def add_slide(title, text=None, data=None):
    slide = ppt.slides.add_slide(ppt.slide_layouts[6])

    # Title
    box = slide.shapes.add_textbox(
        Inches(0.6), Inches(0.3), Inches(12), Inches(0.7)
    )
    p = box.text_frame.paragraphs[0]
    p.text = title
    p.font.size = Pt(28)
    p.font.bold = True

    # Text
    if text:
        box = slide.shapes.add_textbox(
            Inches(0.8), Inches(1.4), Inches(11.5), Inches(4.5)
        )
        box.text_frame.text = text
        for p in box.text_frame.paragraphs:
            p.font.size = Pt(18)

    # Table
    if data is not None:
        rows = len(data) + 1
        cols = len(data.columns)

        table = slide.shapes.add_table(
            rows, cols,
            Inches(0.6), Inches(1.4),
            Inches(12), Inches(5)
        ).table

        for j, col in enumerate(data.columns):
            table.cell(0, j).text = str(col)

        for i in range(len(data)):
            for j in range(cols):
                table.cell(i+1, j).text = str(data.iloc[i, j])

        for row in table.rows:
            for cell in row.cells:
                for p in cell.text_frame.paragraphs:
                    p.font.size = Pt(11)

# Slide 1
add_slide(
    "Project 3: SQL Data Analysis",
    f"Dataset: {len(df)} orders\n"
    f"Columns: {len(df.columns)}\n\n"
    "SQL: SELECT, WHERE, ORDER BY, GROUP BY, COUNT, SUM and AVG"
)

# Slide 2
add_slide(
    "SQL SELECT",
    """SELECT OrderID, Product, Quantity, UnitPrice, TotalPrice
FROM orders
LIMIT 10;"""
)

# Slide 3
add_slide(
    "SQL WHERE + ORDER BY",
    """SELECT OrderID, Product, TotalPrice
FROM orders
WHERE TotalPrice > 2000
ORDER BY TotalPrice DESC;"""
)

# Slide 4
add_slide(
    "Product Analysis",
    data=product
)

# Slide 5
add_slide(
    "Order Status Analysis",
    data=status
)

# Slide 6
add_slide(
    "Payment Method Analysis",
    data=payment
)

# Slide 7
add_slide(
    "Referral Source Analysis",
    data=referral
)

# Slide 8
top = product.iloc[0]

add_slide(
    "Key Insights",
    f"""Highest revenue product: {top['Product']}
Revenue: ${top['Revenue']:,.2f}

Total Orders: {len(df)}

The SQL analysis identifies patterns in products,
order status, payment methods and referral sources."""
)

# Slide 9
add_slide(
    "Conclusion",
    "SQL was used to filter, sort, group and aggregate "
    "the e-commerce dataset to produce useful business insights."
)

# Save
output = "Project_3_SQL_Analysis.pptx"
ppt.save(output)

# Download
files.download(output)

print("PowerPoint created successfully!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

PowerPoint created successfully!
